In [1]:
import torch
import numpy as np
from tqdm import tqdm
import random

import os
os.chdir("..")

from src.models.exogenous_transformer import ExogenousTransformer
from src.shap import get_owen_masks
from src.data import load_data

## Define the settings

In [2]:
FEATURES = [
    "load",
    "hourofday",
    "dayofweek",
    "month",
    "holiday",
    "temperature",
    "precipitation"
]

CATEGORIES = [None, 24, 7, 12, 2, None, None]

EPOCHS = 10
STEPS = 1000
DEVICE = "cuda"

## Create the model

In [3]:
model = ExogenousTransformer(
    encoder_feature_names=FEATURES,
    encoder_categories=CATEGORIES,
    decoder_feature_names=FEATURES[1:],
    decoder_categories=CATEGORIES[1:],
    d_model=32,
    n_layers=1,
    n_heads=1
)
model.to(DEVICE)
print(model)

ExogenousTransformer(
  (encoder_embedding): FeatureEmbedding(
    (embeddings): ParameterDict(
        (load): Object of type: LinearEmbedding
        (hourofday): Object of type: CategoricalEmbedding
        (dayofweek): Object of type: CategoricalEmbedding
        (month): Object of type: CategoricalEmbedding
        (holiday): Object of type: CategoricalEmbedding
        (temperature): Object of type: LinearEmbedding
        (precipitation): Object of type: LinearEmbedding
      (load): LinearEmbedding(
        (embedding): Linear(in_features=1, out_features=32, bias=True)
      )
      (hourofday): CategoricalEmbedding(
        (embedding): Embedding(24, 32)
      )
      (dayofweek): CategoricalEmbedding(
        (embedding): Embedding(7, 32)
      )
      (month): CategoricalEmbedding(
        (embedding): Embedding(12, 32)
      )
      (holiday): CategoricalEmbedding(
        (embedding): Embedding(2, 32)
      )
      (temperature): LinearEmbedding(
        (embedding): Linea

## Load the training data

In [4]:
train_data = load_data("data/transnet/train.json", device=DEVICE)

loss_function = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

n_features = len(FEATURES)

## Generate all feature subset masks for masked training

In [5]:
masks = get_owen_masks(len(FEATURES))
mask_keys = list(masks)

## Training loop

In [6]:
indices = list(range(len(train_data)))

for epoch in range(EPOCHS):
    np.random.shuffle(indices)
    total_loss = 0
    for i in tqdm(indices[:STEPS]):
        optimizer.zero_grad()

        sample = train_data[i]
        x_enc = sample["x_enc"]
        x_dec = sample["x_dec"]
        y = sample["y"]

        x_enc = {k: v[None, :] for k, v in x_enc.items()}
        x_dec = {k: v[None, :] for k, v in x_dec.items()}
        y = y[None, :]

        #ft_mask, attn_mask = get_full_masks(n_features, device=DEVICE)
        ft_mask, attn_mask = masks[random.choice(mask_keys)]

        output = model.forward(x_enc, x_dec, mask_enc=ft_mask[None, :], mask_dec=ft_mask[None, 1:], attn_mask=attn_mask[None, :, :])

        loss = loss_function(output, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {total_loss / STEPS}")

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:11<00:00, 87.21it/s]


Epoch 1/10, Loss: 0.7093335508480668


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:11<00:00, 89.80it/s]


Epoch 2/10, Loss: 0.5875892219804227


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:11<00:00, 90.82it/s]


...

Epoch 8/10, Loss: 0.343977478377521


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:10<00:00, 95.18it/s]


Epoch 9/10, Loss: 0.34009251908026633


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:10<00:00, 97.50it/s]

Epoch 10/10, Loss: 0.33368668910302224


## Save the trained model

In [7]:
# torch.save(model, "exogenous_transformer.pt")